# Finland Electricity Price Analysis — Data Cleaning

This notebook prepares the raw data for analysis. It combines yearly exports into 
single continuous files, handles missing values, parses timestamps, and fixes data 
types — producing three clean, analysis-ready CSVs saved to `data/processed/`.

## 1. Generation Mix (ENTSO-E — Actual Generation per Production Type)

**Source:** ENTSO-E Transparency Platform, report 16.1.B&C, bidding zone BZN|FI  
**Coverage:** 2018–2026, hourly resolution  
**What's in it:** MW output broken down by production type (nuclear, wind, hydro, fossil, etc.)  
**Why we need it:** Main explanatory variable — testing whether generation mix shifts explain price movements  
**Output file:** `finland_generation_mix_2018_2026.csv`

In [1]:
import pandas as pd
import glob

In [4]:
# find every file matching the pattern "AGGREGATED_*.csv" in data folder
file_list = glob.glob('../data/raw/Actual_Generation_per_Production_Type/AGGREGATED_*.CSV')
print(file_list)

['../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201712312300-201812312300 (1).csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201812312300-201912312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201912312300-202012312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202012312300-202112312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202112312300-202212312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202212312300-202312312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202312312300-202412312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202412312300-202512

In [5]:
# loop through each filename, read it into a table, and collect them in a list
all_years = []
for file in file_list:
    df_year = pd.read_csv(file, sep=",")   # sep=";" because your ENTSO-E exports are semicolon-delimited
    all_years.append(df_year)

C:\Users\abhis\AppData\Local\Temp\ipykernel_11364\2201276604.py:4: DtypeWarning: Columns (0: Generation (MW)) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(file, sep=",")   # sep=";" because your ENTSO-E exports are semicolon-delimited


In [6]:
# stack all 8 tables into one combined table
df_price_combined = pd.concat(all_years, ignore_index=True)

In [7]:
# check for duplicate timestamps and product type before trusting the result
duplicates = df_price_combined[df_price_combined.duplicated(subset=["MTU (CET/CEST)", "Production Type"], keep=False)]
print("Number of duplicate rows:", len(duplicates))

Number of duplicate rows: 0


In [8]:
df_price_combined.head()

,MTU (CET/CEST),Area,Production Type,Generation (MW)
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,BZN|FI,Biomass,873.31
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,BZN|FI,Biomass,855.71
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,BZN|FI,Biomass,858.29
3,01/01/2018 03:00:00 - 01/01/2018 04:00:00,BZN|FI,Biomass,874.48
4,01/01/2018 04:00:00 - 01/01/2018 05:00:00,BZN|FI,Biomass,869.99


In [9]:
# Rename columns to clean, consistent snake_case names for easier analysis
df_price_combined.columns = ['timestamp_raw', 'area', 'production_type','generation_mw']

In [10]:
# Null value check
df_price_combined.isnull().sum()

timestamp_raw       0
area                0
production_type     0
generation_mw      40
dtype: int64

In [11]:
# Identify which production types have null values
print(df_price_combined[df_price_combined['generation_mw'].isnull()]['production_type'].value_counts())

production_type
Solar                             12
Fossil Gas                         3
Fossil Hard coal                   3
Fossil Oil                         3
Hydro Run-of-river and pondage     3
Other renewable                    3
Waste                              3
Biomass                            2
Fossil Peat                        2
Nuclear                            2
Other                              2
Wind Onshore                       2
Name: count, dtype: int64


In [12]:
# Sort first so forward fill works in the right direction
df_price_combined = df_price_combined.sort_values(['production_type', 'timestamp_raw'])

In [13]:
# Forward fill nulls within each production type separately
df_price_combined['generation_mw'] =df_price_combined.groupby('production_type')['generation_mw'].ffill()

In [14]:
# Confirm no nulls remain
df_price_combined['generation_mw'].isnull().sum()

np.int64(0)

In [15]:
# Extract start time from the timestamp range string (e.g. "01/01/2018 00:00:00 - 01/01/2018 01:00:00")
# Only the start time is needed — end time is always exactly one hour later and carries no additional information
df_price_combined['timestamp'] = df_price_combined['timestamp_raw'].str.split("-").str[0]

In [16]:
df_price_combined['timestamp']

0          01/01/2018 00:00:00 
1          01/01/2018 01:00:00 
2          01/01/2018 02:00:00 
3          01/01/2018 03:00:00 
4          01/01/2018 04:00:00 
                   ...         
3313795    31/12/2026 22:45:00 
3313796    31/12/2026 23:00:00 
3313797    31/12/2026 23:15:00 
3313798    31/12/2026 23:30:00 
3313799    31/12/2026 23:45:00 
Name: timestamp, Length: 3313800, dtype: object

In [17]:
# Strip any timezone suffix like (CET), (CEST), etc.
df_price_combined['timestamp'] = (
    df_price_combined['timestamp']
    .str.replace(r'\s*\([^)]*\)', '', regex=True)  # removes anything in parentheses
    .str.strip()
)

In [18]:
# Convert to datetime
df_price_combined['timestamp'] = pd.to_datetime(
    df_price_combined['timestamp'],
    format='%d/%m/%Y %H:%M:%S',
    dayfirst=True
)

In [19]:
# Drop timestamp_raw now that clean datetime column has been created
df_price_combined =df_price_combined.drop(columns='timestamp_raw')

In [20]:
# Replace n/e and - with 0, then convert to float
df_price_combined['generation_mw'] = df_price_combined['generation_mw'].replace({'n/e':0, '-':0})

In [21]:
# convert to float
df_price_combined['generation_mw'] = pd.to_numeric(df_price_combined['generation_mw'])

### Final verification before saving

In [22]:
df_price_combined.shape

(3313800, 4)

In [23]:
df_price_combined.dtypes

area                          str
production_type               str
generation_mw             float64
timestamp          datetime64[us]
dtype: object

In [24]:
df_price_combined.isnull().sum()

area               0
production_type    0
generation_mw      0
timestamp          0
dtype: int64

In [25]:
df_price_combined.head()

,area,production_type,generation_mw,timestamp
0,BZN|FI,Biomass,873.31,2018-01-01 00:00:00
1,BZN|FI,Biomass,855.71,2018-01-01 01:00:00
2,BZN|FI,Biomass,858.29,2018-01-01 02:00:00
3,BZN|FI,Biomass,874.48,2018-01-01 03:00:00
4,BZN|FI,Biomass,869.99,2018-01-01 04:00:00


### Save cleaned consumption dataset

In [ ]:

df_price_combined = df_price_combined.sort_values(["timestamp", "production_type"])
df_price_combined.to_csv('../data/processed/finland_generation_mix_2018_2026.csv', index=False)
print("Saved successfully, total rows:", len(df_price_combined))

Saved successfully, total rows: 3313800


## 2. Day-ahead Electricity Price (ENTSO-E — Energy Prices 12.1.D)
**Source:** ENTSO-E Transparency Platform, report 12.1.D, bidding zone BZN|FI  
**Coverage:** 2018–2026, hourly resolution  
**What's in it:** Day-ahead electricity price in EUR/MWh for each hour  
**Why we need it:** Main outcome variable — the price we're trying to explain  

### find every file matching the pattern "AGGREGATED_*.csv" in data folder


In [27]:
# find every file matching the pattern "GUI_*.csv" in data folder

file_list = glob.glob('../data/raw/Energy_Prices/GUI_*.CSV')
print(file_list)

['../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_201712312300-201812312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_201812312300-201912312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_201912312300-202012312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202012312300-202112312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202112312300-202212312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202212312300-202312312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202312312300-202412312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202412312300-202512312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202512312300-202612312300.csv']


In [28]:
# loop through each filename, read it into a table, and collect them in a list
all_years = []
for file in file_list:
    df_year = pd.read_csv(file, sep=",")
    all_years.append(df_year)

In [29]:
# stack all 9 tables into one combined table

df_energy_prices = pd.concat(all_years, ignore_index=True)

In [30]:
df_energy_prices.head()

,MTU (CET/CEST),Area,Sequence,Day-ahead Price (EUR/MWh),Intraday Period (CET/CEST),Intraday Price (EUR/MWh)
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,BZN|FI,Without Sequence,26.33,NaN,NaN
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,BZN|FI,Without Sequence,26.43,NaN,NaN
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,BZN|FI,Without Sequence,26.10,NaN,NaN
3,01/01/2018 03:00:00 - 01/01/2018 04:00:00,BZN|FI,Without Sequence,24.70,NaN,NaN
4,01/01/2018 04:00:00 - 01/01/2018 05:00:00,BZN|FI,Without Sequence,24.74,NaN,NaN


In [31]:
df_energy_prices.shape

(112628, 6)

In [32]:
df_energy_prices.isnull().sum()

MTU (CET/CEST)                     0
Area                               0
Sequence                           0
Day-ahead Price (EUR/MWh)          0
Intraday Period (CET/CEST)    112628
Intraday Price (EUR/MWh)      112628
dtype: int64

In [33]:
df_energy_prices =df_energy_prices.drop(columns=['Intraday Period (CET/CEST)', 'Intraday Price (EUR/MWh)'])

In [34]:
# Rename columns to clean, consistent snake_case names for easier analysis
df_energy_prices.columns = ['timestamp_raw', 'area','sequence' ,'price_eur_mwh']

In [35]:
df_energy_prices.dtypes

timestamp_raw        str
area                 str
sequence             str
price_eur_mwh    float64
dtype: object

In [36]:
# Extract start time from the timestamp range string (e.g. "01/01/2018 00:00:00 - 01/01/2018 01:00:00")
# Only the start time is needed — end time is always exactly one hour later and carries no additional information
df_energy_prices['timestamp'] =df_energy_prices['timestamp_raw'].str.split("-").str[0]

In [37]:
df_energy_prices['timestamp']

0         01/01/2018 00:00:00 
1         01/01/2018 01:00:00 
2         01/01/2018 02:00:00 
3         01/01/2018 03:00:00 
4         01/01/2018 04:00:00 
                  ...         
112623    18/06/2026 22:45:00 
112624    18/06/2026 23:00:00 
112625    18/06/2026 23:15:00 
112626    18/06/2026 23:30:00 
112627    18/06/2026 23:45:00 
Name: timestamp, Length: 112628, dtype: object

In [38]:
# Strip timezone label (CET/CEST) left over after split

df_energy_prices['timestamp'] = df_energy_prices['timestamp'].str.replace(r'\s*\([^)]*\)', '', regex=True).str.strip()

In [39]:
# Convert to datetime

df_energy_prices['timestamp'] = pd.to_datetime(df_energy_prices['timestamp'],
                                               format='%d/%m/%Y %H:%M:%S',
                                               dayfirst=True) 

In [40]:
print(df_energy_prices['timestamp'].dtype)

datetime64[us]


In [41]:
df_energy_prices = df_energy_prices.drop(columns='timestamp_raw')

### Final verification before saving

In [42]:
df_energy_prices.shape

(112628, 4)

In [43]:
df_energy_prices.dtypes

area                        str
sequence                    str
price_eur_mwh           float64
timestamp        datetime64[us]
dtype: object

In [44]:
df_energy_prices.isnull().sum()

area             0
sequence         0
price_eur_mwh    0
timestamp        0
dtype: int64

In [45]:
df_energy_prices.head()

,area,sequence,price_eur_mwh,timestamp
0,BZN|FI,Without Sequence,26.33,2018-01-01 00:00:00
1,BZN|FI,Without Sequence,26.43,2018-01-01 01:00:00
2,BZN|FI,Without Sequence,26.10,2018-01-01 02:00:00
3,BZN|FI,Without Sequence,24.70,2018-01-01 03:00:00
4,BZN|FI,Without Sequence,24.74,2018-01-01 04:00:00


### Save cleaned consumption dataset

In [ ]:

df_energy_prices = df_energy_prices.sort_values('timestamp')
df_energy_prices.to_csv('../data/processed/finland_electricity_price_2018_2026.csv', index=False)
print("Saved successfully, total rows:", len(df_energy_prices))

Saved successfully, total rows: 112628


## 3. Electricity Consumption (Fingrid — Dataset 363)

**Source:** Fingrid Open Data, dataset 363  
**Coverage:** August 2023–2026, hourly resolution  
**What's in it:** Total electricity consumed in Finnish distribution networks, in kWh per hour  
**Why we need it:** Demand-side control variable — ensures wind/price correlations aren't just seasonal demand artifacts  
**Note:** Shorter coverage than other datasets (Aug 2023 onward only) — used for post-crisis period analysis  
**Output file:** `finland_consumption_2023_2026.csv`

In [ ]:
# Load Fingrid consumption dataset (dataset 363)
df_consumption = pd.read_csv('../data/raw/Fingrid Dataset/363_2018-01-01T0100_2026-06-17T0000.csv', sep=';')

In [71]:
df_consumption.head()

,startTime,endTime,Total electricity consumption in Finnish distribution networks,additionalJson
0,2023-08-01T00:00:00.000Z,2023-08-01T01:00:00.000Z,3247514.461,"{""TimeSeriesType"":""CTT_SUM_CONS_ACP"",""Res"":""PT..."
1,2023-08-01T01:00:00.000Z,2023-08-01T02:00:00.000Z,3198878.399,"{""TimeSeriesType"":""CTT_SUM_CONS_ACP"",""Res"":""PT..."
2,2023-08-01T02:00:00.000Z,2023-08-01T03:00:00.000Z,3276928.820,"{""TimeSeriesType"":""CTT_SUM_CONS_ACP"",""Res"":""PT..."
3,2023-08-01T03:00:00.000Z,2023-08-01T04:00:00.000Z,3771434.303,"{""TimeSeriesType"":""CTT_SUM_CONS_ACP"",""Res"":""PT..."
4,2023-08-01T04:00:00.000Z,2023-08-01T05:00:00.000Z,4199003.055,"{""TimeSeriesType"":""CTT_SUM_CONS_ACP"",""Res"":""PT..."


In [90]:
# Drop redundant and unnecessary column
df_consumption = df_consumption.drop(columns=['endTime', 'additionalJson'])

In [91]:
# Rename columns to clean snake_case names
df_consumption.columns = ['timestamp_raw', 'consumption_kwh']

In [92]:
df_consumption.shape

(25176, 2)

In [93]:
df_consumption.isnull().sum()

timestamp_raw      0
consumption_kwh    0
dtype: int64

In [94]:
df_consumption.dtypes

timestamp_raw          str
consumption_kwh    float64
dtype: object

In [95]:
# Parse timestamp from ISO 8601 UTC format

df_consumption['timestamp'] = pd.to_datetime(df_consumption['timestamp_raw'], utc=True)

In [96]:
df_consumption['timestamp'].dtypes

datetime64[us, UTC]

In [97]:
# Convert consumption from kWh to MWh to match units used in price and generation datasets
# 1 MWh = 1000 kWh

df_consumption['consumption_mwh'] = df_consumption['consumption_kwh'] / 1000

In [99]:
# Drop redundant and unnecessary column now that clean versions exist

df_consumption = df_consumption.drop(columns=['timestamp_raw','consumption_kwh'])

In [100]:
df_consumption.columns

Index(['timestamp', 'consumption_mwh'], dtype='str')

In [102]:

print(df_consumption['consumption_mwh'].describe())

count    25176.000000
mean      5864.295648
std       1477.658380
min       2996.717151
25%       4750.552195
50%       5689.018921
75%       6879.622426
max      10813.838488
Name: consumption_mwh, dtype: float64


### Final verification before saving

In [103]:
df_consumption.shape

(25176, 2)

In [104]:
df_consumption.dtypes

timestamp          datetime64[us, UTC]
consumption_mwh                float64
dtype: object

In [105]:
df_consumption.isnull().sum()

timestamp          0
consumption_mwh    0
dtype: int64

In [106]:
df_consumption.head()

,timestamp,consumption_mwh
0,2023-08-01 00:00:00+00:00,3247.514461
1,2023-08-01 01:00:00+00:00,3198.878399
2,2023-08-01 02:00:00+00:00,3276.928820
3,2023-08-01 03:00:00+00:00,3771.434303
4,2023-08-01 04:00:00+00:00,4199.003055


### Save cleaned consumption dataset

In [107]:
df_consumption = df_consumption.sort_values('timestamp')
df_consumption.to_csv('../data/processed/finland_consumption_2023_2026.csv', index=False)
print("Saved successfully, total rows:", len(df_consumption))

Saved successfully, total rows: 25176


## 4. European Reference Prices — Germany (Ember Climate)

**Source:** Ember Climate, European Wholesale Electricity Price Data  
**Coverage:** 2018–2026, hourly resolution  
**What's in it:** Day-ahead wholesale electricity price in EUR/MWh for Germany  
**Why we need it:** External factor reference — shows whether Finnish price movements followed the broader European market during the 2022 crisis, or were driven by local factors  
**Note:** Germany is the largest European electricity market and was heavily gas-dependent in 2022, making it the best proxy for the European gas-crisis-driven price spike  
**Output file:** `germany_electricity_price_2018_2026.csv`

In [ ]:
# Load Germany day-ahead electricity price data from Ember Climate
df_germany_energy_prices = pd.read_csv('../data/raw/European_Reference_Prices/Germany.csv', sep=',')

In [115]:
df_germany_energy_prices.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe)
0,Germany,DEU,2015-01-01 00:00:00,2015-01-01 01:00:00,22.34
1,Germany,DEU,2015-01-01 01:00:00,2015-01-01 02:00:00,22.34
2,Germany,DEU,2015-01-01 02:00:00,2015-01-01 03:00:00,22.34
3,Germany,DEU,2015-01-01 03:00:00,2015-01-01 04:00:00,22.34
4,Germany,DEU,2015-01-01 04:00:00,2015-01-01 05:00:00,22.34


In [110]:
df_germany_energy_prices.shape

(102260, 5)

In [111]:
df_germany_energy_prices.isnull().sum()

Country             0
ISO3 Code           0
Datetime (UTC)      0
Datetime (Local)    0
Price (EUR/MWhe)    0
dtype: int64

In [112]:
df_germany_energy_prices.dtypes

Country                 str
ISO3 Code               str
Datetime (UTC)          str
Datetime (Local)        str
Price (EUR/MWhe)    float64
dtype: object

In [ ]:
# Drop redundant and unnecessary columns
df_germany_energy_prices = df_germany_energy_prices.drop(columns= ['Country', 'ISO3 Code', 'Datetime (Local)'])

In [118]:
df_germany_energy_prices.head()

,Datetime (UTC),Price (EUR/MWhe)
0,2015-01-01 00:00:00,22.34
1,2015-01-01 01:00:00,22.34
2,2015-01-01 02:00:00,22.34
3,2015-01-01 03:00:00,22.34
4,2015-01-01 04:00:00,22.34


In [ ]:
# Rename columns to clean snake_case names
df_germany_energy_prices.columns = ['timestamp', 'price_eur_mwh']

In [ ]:
# Parse timestamp to datetime with UTC timezone
df_germany_energy_prices['timestamp'] =pd.to_datetime(df_germany_energy_prices['timestamp'], utc=True)

In [122]:
df_germany_energy_prices['timestamp'].dtypes

datetime64[us, UTC]

In [ ]:
# Filter to 2018 onward to align with Finnish price and generation mix datasets
# Raw file starts from 2015 but pre-2018 data is not needed for this analysis
df_germany_energy_prices =df_germany_energy_prices[df_germany_energy_prices['timestamp'] >= '2018-01-01']

### Final verification before saving

In [125]:
df_germany_energy_prices.shape

(75956, 2)

In [126]:
df_germany_energy_prices.dtypes

timestamp        datetime64[us, UTC]
price_eur_mwh                float64
dtype: object

In [127]:
df_germany_energy_prices.isnull().sum()

timestamp        0
price_eur_mwh    0
dtype: int64

In [129]:
print("Start:", df_germany_energy_prices['timestamp'].min())
print("End:", df_germany_energy_prices['timestamp'].max())

Start: 2018-01-01 00:00:00+00:00
End: 2026-08-31 21:00:00+00:00


In [128]:
df_germany_energy_prices.head()

,timestamp,price_eur_mwh
26304,2018-01-01 00:00:00+00:00,-29.99
26305,2018-01-01 01:00:00+00:00,-56.65
26306,2018-01-01 02:00:00+00:00,-63.14
26307,2018-01-01 03:00:00+00:00,-64.62
26308,2018-01-01 04:00:00+00:00,-67.00


### Save cleaned consumption dataset

In [130]:
df_germany_energy_prices = df_germany_energy_prices.sort_values('timestamp')
df_germany_energy_prices.to_csv('../data/processed/germany_electricity_price_2018_2026.csv', index=False)
print("Saved successfully, total rows:", len(df_germany_energy_prices))

Saved successfully, total rows: 75956


## 5. European Reference Prices — Sweden (Ember Climate)

**Source:** Ember Climate, European Wholesale Electricity Price Data  
**Coverage:** 2018–2026, hourly resolution  
**What's in it:** Day-ahead wholesale electricity price in EUR/MWh for Sweden  
**Why we need it:** Nordic regional reference — Sweden shares the same Nord Pool market as Finland, so price divergence between Finland and Sweden around May 2022 points to Finland-specific factors (Russian import cutoff) rather than broader European market movements  
**Output file:** `sweden_electricity_price_2018_2026.csv`

In [2]:
df_sweden_energy_prices = pd.read_csv('../data/raw/European_Reference_Prices/Sweden.csv', sep=',')

### Initial inspection
Quick check of structure, shape, data types, and null values before cleaning.

In [3]:
df_sweden_energy_prices.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe)
0,Sweden,SWE,2015-01-01 00:00:00,2015-01-01 01:00:00,23.37
1,Sweden,SWE,2015-01-01 01:00:00,2015-01-01 02:00:00,19.33
2,Sweden,SWE,2015-01-01 02:00:00,2015-01-01 03:00:00,17.66
3,Sweden,SWE,2015-01-01 03:00:00,2015-01-01 04:00:00,17.53
4,Sweden,SWE,2015-01-01 04:00:00,2015-01-01 05:00:00,18.07


In [4]:
df_sweden_energy_prices.shape

(102169, 5)

In [5]:
df_sweden_energy_prices.isnull().sum()

Country                0
ISO3 Code              0
Datetime (UTC)         0
Datetime (Local)       0
Price (EUR/MWhe)    2880
dtype: int64

In [7]:
df_sweden_energy_prices.dtypes

Datetime (UTC)          str
Price (EUR/MWhe)    float64
dtype: object

In [ ]:
# Drop redundant and unnecessary columns
df_sweden_energy_prices = df_sweden_energy_prices.drop(columns= ['Country', 'ISO3 Code', 'Datetime (Local)'])

In [ ]:
# Rename columns to clean snake_case names
df_sweden_energy_prices.columns = ['timestamp', 'price_eur_mwh']

In [ ]:
# Parse timestamp to datetime with UTC timezone
df_sweden_energy_prices['timestamp'] = pd.to_datetime(df_sweden_energy_prices['timestamp'], utc= True)

In [ ]:
# Verify timestamp parsed correctly
df_sweden_energy_prices['timestamp'].dtype

datetime64[us, UTC]

In [ ]:
# Filter to 2018 onward to align with Finnish price and generation mix datasets
# Raw file starts from 2015 but pre-2018 data is not needed for this analysis
df_sweden_energy_prices = df_sweden_energy_prices[df_sweden_energy_prices['timestamp'] >= '2018-01-01']

In [13]:
df_sweden_energy_prices.head()

,timestamp,price_eur_mwh
26304,2018-01-01 00:00:00+00:00,26.43
26305,2018-01-01 01:00:00+00:00,26.10
26306,2018-01-01 02:00:00+00:00,24.70
26307,2018-01-01 03:00:00+00:00,24.74
26308,2018-01-01 04:00:00+00:00,18.01


In [ ]:
# Checking for missing values after filtering to 2018 onward.
df_sweden_energy_prices.isnull().sum()

timestamp           0
price_eur_mwh    2880
dtype: int64

In [ ]:
# Identify which years contain null price values
# helps determine whether nulls are a recent data gap or spread across the full series
print(df_sweden_energy_prices[df_sweden_energy_prices['price_eur_mwh'].isnull()]['timestamp'].dt.year.value_counts().sort_index())

timestamp
2026    2880
Name: count, dtype: int64


In [17]:
# Drop 2026 rows — Sweden price data not yet available for this period in Ember dataset
df_sweden_energy_prices = df_sweden_energy_prices.dropna(subset=['price_eur_mwh'])


In [18]:
# Confirm no nulls remain
print(df_sweden_energy_prices['price_eur_mwh'].isnull().sum())

0


### Final verification before saving

In [19]:
df_sweden_energy_prices.shape

(72985, 2)

In [20]:
df_sweden_energy_prices.dtypes

timestamp        datetime64[us, UTC]
price_eur_mwh                float64
dtype: object

In [21]:
df_sweden_energy_prices.isnull().sum()

timestamp        0
price_eur_mwh    0
dtype: int64

In [22]:
df_sweden_energy_prices.head()

,timestamp,price_eur_mwh
26304,2018-01-01 00:00:00+00:00,26.43
26305,2018-01-01 01:00:00+00:00,26.10
26306,2018-01-01 02:00:00+00:00,24.70
26307,2018-01-01 03:00:00+00:00,24.74
26308,2018-01-01 04:00:00+00:00,18.01


### Save cleaned consumption dataset

In [24]:
df_sweden_energy_prices = df_sweden_energy_prices.sort_values('timestamp')
df_sweden_energy_prices.to_csv('../data/processed/sweden_electricity_price_2018_2026.csv', index= False)
print("Saved successfully, total rows:", len(df_sweden_energy_prices))

Saved successfully, total rows: 72985
